# Nesne Tespiti: YOLO ve SSD\nYOLOv8 ve SSD ile gerçek zamanlı nesne tespiti. COCO128 veri seti ile eğitim, şehir/sokak görsellerinde çıkarım.\n\nİstanbul sokak fotoğrafı ile test: `img = 'fotograflar/istanbul_sokak.jpg'` olarak değiştir.

In [ ]:
import cv2, os, numpy as np, matplotlib.pyplot as plt\nfrom pathlib import Path\nfrom PIL import Image\nos.makedirs('cikti', exist_ok=True)

## YOLOv8 Nano ile Nesne Tespiti

In [ ]:
from ultralytics import YOLO\n\nCOCO_SINIFLAR = ['insan','bisiklet','araba','motorsiklet','ucak','otobus','tren','kamyon','bot','trafik_isigi']\n\nmodel = YOLO('yolov8n.pt')\nprint('YOLOv8n yüklendi.')\n\n# Demo resmi oluştur (kendi fotoğrafın için yolu değiştir)\nimg = Image.new('RGB', (640, 480), (100, 100, 100))\nimg.save('cikti/test.jpg')\n\nresults = model('cikti/test.jpg')\nresult = results[0]\nboxes = result.boxes\nif boxes is not None and len(boxes) > 0:\n    print(f'Tespit edilen {len(boxes)} nesne:')\n    for i, box in enumerate(boxes):\n        cls_id = int(box.cls[0].item())\n        conf = box.conf[0].item()\n        name = COCO_SINIFLAR[cls_id] if cls_id < len(COCO_SINIFLAR) else f'sinif_{cls_id}'\n        print(f'  [{i+1}] {name} (güven: {conf:.2f})')\nelse:\n    print('Nesne tespit edilemedi.')\n\nresult.save('cikti/yolo_sonuc.jpg')\nprint('Çıktı: cikti/yolo_sonuc.jpg')

## SSD (Single Shot MultiBox Detector)

In [ ]:
import torch\nfrom torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights\nfrom torchvision.transforms import functional as F\n\nweights = SSDLite320_MobileNet_V3_Large_Weights.DEFAULT\nmodel = ssdlite320_mobilenet_v3_large(weights=weights)\nmodel.eval()\nprint('SSD modeli yüklendi.')\n\nimg = Image.new('RGB', (320, 320), (100, 120, 100))\ntensor = F.to_tensor(img).unsqueeze(0)\n\nwith torch.no_grad():\n    pred = model(tensor)\n    \nscores = pred[0]['scores']\nkeep = scores > 0.5\nprint(f'SSD tespit: {keep.sum().item()} nesne (skor > 0.5)')

## COCO128 ile YOLO Eğitimi (Demo)

In [ ]:
# 2 epoch, demo amaçlı. Gerçek eğitim için epoch=50-100 önerilir.\ntry:\n    model.train(data='coco128.yaml', epochs=2, imgsz=320, verbose=False, exist_ok=True)\n    print('Eğitim tamamlandı. Sonuçlar: runs/detect/train/')\nexcept Exception as e:\n    print(f'Eğitim atlandı: {e}')